# One slot machine: how does your belief change as you play?
**Teams of 2–3**

You are sitting in front of one slot machine. You do not know its chance of winning. You will choose what you believe before playing, reveal a batch of results, and update your belief.

There is **one machine with one fixed, unknown win probability**, which we call $p$. The machine does not change during the exercise. Our knowledge of it changes as we observe more plays.

Each play costs **1 token**. A win pays **2.4 tokens total**, so you gain 1.4 tokens. A loss costs your token.

At each **Pause and predict**, discuss your answer before running the next cell.


In [ ]:
# Setup only: import libraries and configure paths
from pathlib import Path
import sys, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'workshop').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.spines.top': False, 'axes.spines.right': False})
DATA = ROOT / 'data' / 'public'


## 1. What would you believe before playing?
If $p = 0.4$, each play has a 40% chance of winning. That does not mean exactly four of the next ten plays must win: short runs vary just by chance.

Before seeing any results, what values of $p$ seem plausible? Perhaps you know very little about the machine. Perhaps you think a casino usually sets its games up so the house makes money.

A **prior** describes this starting belief as a range of possible $p$ values. It expresses both *where you think $p$ might be* and *how confident you are*. It is an assumption, not a record of plays that really happened.

For our payout, the break-even probability is $1 / 2.4$, about **0.417**. A smaller $p$ loses tokens on average; a larger $p$ gains tokens on average. A lucky or unlucky short run is still possible on either side.

We will compare three starting beliefs. All use the **Beta distribution**, a family of curves for probabilities between zero and one.

| Choice | Starting belief |
|---|---|
| Broad: Beta(1, 1) | Give equal density to all $p$ values between 0 and 1. This represents "Laplace's principle of indifference" and is often called an "uninformed prior". |
| Mild house edge: Beta(2, 3) | Lean toward a win probability around 0.4 (the break even rate), but with substantial uncertainty.|
| Las Vegas: Beta(38, 62) | The house probably has an edge, because they want to profit, but not too much of an edge that gamblers suspect a rigged game. |

**Pause and predict:** which prior would change the least after observing a few wins?


In [ ]:
from scipy.stats import beta, binom
import importlib
import workshop.casino as casino
# Pick up helper edits even when this notebook kernel was already running.
importlib.reload(casino)
from workshop.casino import PRIOR_CHOICES, fit_slot, simulate_profit
p_grid = np.linspace(.001, .999, 600)
break_even = 1 / 2.4
rows = []
for name, (aa, bb) in PRIOR_CHOICES.items():
    plt.plot(p_grid, beta.pdf(p_grid, aa, bb), label=name)
    rows.append({'Prior': name, 'Mean p': aa/(aa+bb),
                 '90% low': beta.ppf(.05, aa, bb), '90% high': beta.ppf(.95, aa, bb),
                 'Probability p is profitable': beta.sf(break_even, aa, bb)})
plt.axvline(break_even, color='tomato', linestyle='--', label='Break even')
plt.xlabel('Possible win probability p'); plt.ylabel('Density'); plt.legend(); plt.show()
display(pd.DataFrame(rows).round(3))


The horizontal axis lists possible answers for $p$. The **area under the curve** for a small region is the probability of the win rate in that region. In other words, a higher area for an interval $\Delta p$ means more belief that $p$ lies in that region. The height at a single point is not a probability; a narrow curve can have a height greater than one.

The “90% low” and “90% high” columns enclose the middle 90% of each prior's probability. Notice that even the Las Vegas prior allows a profitable machine. “The house probably wins” is different from “a profitable machine is impossible.”

### Choose your starting belief
Select a prior below and tell your teammates why. We will use it in every later step. To change it, return here and rerun the cells below.


In [ ]:
# Select a prior
# Choose 'Broad', 'Mild house edge', or 'Las Vegas'

# SELECTED_PRIOR = 'Broad'
# SELECTED_PRIOR = 'Mild house edge'
SELECTED_PRIOR = 'Las Vegas'

a, b = PRIOR_CHOICES[SELECTED_PRIOR]
print(f'Our prior is {SELECTED_PRIOR}: Beta({a:g}, {b:g}).')
print(f'Before seeing any results, our mean win probability is {a/(a+b):.1%}.')


### What do the Beta numbers mean?
A Beta prior has two positive numbers, $a$ and $b$. Their ratio $a / (a+b)$ gives the prior mean. Keeping that ratio fixed while increasing their total $a+b$ makes the belief more concentrated.

Compare Beta(2, 3) and Beta(20, 30). Both have mean 0.4. The second expresses greater confidence in that neighborhood. This separates *what you believe* from *how strongly you believe it*.

**Pause and predict:** which curve will be narrower?


In [ ]:
for aa, bb in [(2, 3), (20, 30)]:
    plt.plot(p_grid, beta.pdf(p_grid, aa, bb), label=f'Beta({aa}, {bb})')
plt.xlabel('Possible win probability p'); plt.ylabel('Density'); plt.legend(); plt.show()


## 2. Play the machine and look at the results
A file in the repo contains one reproducible sequence of plays from our machine.

Start with **five plays**. After working through the next sections, you will be instructed to return here and change `PLAYS_SEEN` to 20, then 100, and finally 500. Rerun the cells below each time.

**Pause and predict:** how confident would you be after only five plays? If the first few plays all won, would that establish $p = 1$?


In [ ]:
all_plays = pd.read_csv(DATA / 'slot_history.csv')
PLAYS_SEEN = 5  # Continue the SAME run: try 20, 100, and 500 next.
if not isinstance(PLAYS_SEEN, int) or not 0 <= PLAYS_SEEN <= len(all_plays):
    raise ValueError(f'Choose a whole number from 0 to {len(all_plays)}.')
observed = all_plays.iloc[:PLAYS_SEEN].copy()
plays = len(observed)
wins = int(observed.win.sum())
losses = plays - wins
print(f'We have seen {wins} wins and {losses} losses in {plays} plays.')
if plays:
    print(f'Observed win fraction: {wins/plays:.1%}')
    plt.scatter(observed.play, observed.win, s=20, alpha=.65)
    plt.yticks([0, 1], ['Loss', 'Win']); plt.xlabel('Play number'); plt.title('Results revealed so far'); plt.show()
else:
    print('No observations yet: our belief is still the prior.')


## 3. Which possible $p$ values explain these results?
Suppose a machine won three times in three plays. If its $p$ were 0.4, that run would have probability $0.4 × 0.4 × 0.4 = 0.064$. It is unusual, but possible. A machine with a higher $p$ would make that run more likely.

For our actual observations, we can calculate the probability of seeing this win count under each possible $p$. This is the **likelihood**: it tells us how well each candidate $p$ explains the data.

It is not yet our updated belief about $p$. It only scores the evidence; we still need to combine it with the prior.

**Pause and predict:** where should the likelihood be largest for the win fraction you just observed?


In [ ]:
likelihood = binom.pmf(wins, plays, p_grid)
plt.plot(p_grid, likelihood, color='darkorange')
plt.xlabel('Candidate win probability p'); plt.ylabel('Probability of the observed win count')
plt.title(f'The evidence: {wins} wins in {plays} plays'); plt.show()


## 4. Update your belief: the posterior
The **posterior** is our belief about $p$ after taking the observations into account. It combines what we believed beforehand with what the data tell us.

Bayes' rule says:

**posterior density ∝ prior density × likelihood**

The symbol ∝ means “proportional to.” We multiply the two curves and rescale the result so its total area is one. Possible $p$ values need support from both our prior and the evidence to receive much posterior weight.

For a Beta prior with win/loss data, there is a convenient exact update:

$\mathrm{Beta}(a,b) \rightarrow \mathrm{Beta}(a+\text{wins},\ b+\text{losses})$

For example, three wins and no losses turn Beta(2, 3) into Beta(5, 3). The same observations turn the stronger Las Vegas prior into Beta(41, 62). The prior numbers act like starting weights in this calculation; they are not real plays.

**Pause and predict:** should these two starting beliefs lead to identical conclusions after only three observations?


In [ ]:
a, b = PRIOR_CHOICES[SELECTED_PRIOR]
post_a, post_b = a + wins, b + losses

from matplotlib.ticker import PercentFormatter
prior_colors = dict(zip(PRIOR_CHOICES, ['#8064A2', '#2878B5', '#22856A']))
posterior_color = prior_colors[SELECTED_PRIOR]

# Scale the likelihood to area one for comparison with the two densities.
# This does not make the likelihood a posterior distribution.
prior_density = beta.pdf(p_grid, a, b)
scaled_likelihood = beta.pdf(p_grid, wins + 1, losses + 1)
posterior_density = beta.pdf(p_grid, post_a, post_b)

fig, ax = plt.subplots(figsize=(12, 5.5), layout='constrained')
ax.plot(p_grid, prior_density, color='#64748B', linestyle='--',
        linewidth=2, label=f'Prior: {SELECTED_PRIOR}')
ax.plot(p_grid, scaled_likelihood, color='#D58A22', linestyle='-.',
        linewidth=2, label='Likelihood (scaled)')
ax.plot(p_grid, posterior_density, color=posterior_color,
        linewidth=3, label='Posterior')
ax.fill_between(p_grid, posterior_density, color=posterior_color, alpha=.09)
ax.axvline(break_even, color='#C05252', linestyle=':', linewidth=1.8,
           label=f'Break even ({break_even:.1%})')
ax.set(xlim=(0, 1), ylim=(0, None), xlabel='Win probability p',
       ylabel='Density / scaled likelihood')
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.set_title(f'Updating our belief: {wins} wins and {losses} losses',
             loc='left', fontsize=15, pad=15)
ax.grid(axis='y', alpha=.15)
ax.set_axisbelow(True)
ax.legend(loc='upper center', bbox_to_anchor=(.5, -.15), ncol=2, frameon=False)
plt.show()

print('The likelihood is scaled to area one to compare shapes; it is not itself our posterior.')
print(f'Prior mean p: {a/(a+b):.1%}; posterior mean p: {post_a/(post_a+post_b):.1%}')
interval = beta.ppf([.05, .95], post_a, post_b)
print(f'90% posterior interval for p: {interval[0]:.1%} to {interval[1]:.1%}')
print(f'Posterior probability of a profitable p: {beta.sf(break_even, post_a, post_b):.1%}')


The posterior is a distribution over $p$, not a claim that we have discovered its exact value. Its mean is one summary; the interval tells us how much uncertainty remains. You might also notice, particularly for cases with a low number of samples, that the posterior distribution acts as a sort of "middle ground" between the prior and likelihood distributions. 

### Would another prior tell a different story?
Every panel below uses the **same machine and exactly the same observations**. Only the starting belief differs. The dashed curve is the prior, and the solid curve is the posterior.

If the posteriors disagree, the evidence may not yet be strong enough to overcome their different starting assumptions. That is useful information about how much confidence to place in a decision.


In [ ]:
from matplotlib.ticker import PercentFormatter
prior_colors = dict(zip(PRIOR_CHOICES, ['#8064A2', '#2878B5', '#22856A']))

# Three individual updates above, one shared posterior comparison below.
fig = plt.figure(figsize=(13, 8.5), layout='constrained')
grid = fig.add_gridspec(2, 3, height_ratios=[1, 1.35], hspace=.12)
top_axes = [fig.add_subplot(grid[0, 0])]
for column in [1, 2]:
    top_axes.append(fig.add_subplot(grid[0, column], sharex=top_axes[0], sharey=top_axes[0]))
combined = fig.add_subplot(grid[1, :])
comparison = []

for ax, (name, (aa, bb)) in zip(top_axes, PRIOR_CHOICES.items()):
    color = prior_colors[name]
    posterior = beta.pdf(p_grid, aa + wins, bb + losses)
    ax.plot(p_grid, beta.pdf(p_grid, aa, bb), color=color,
            linestyle='--', linewidth=1.8)
    ax.plot(p_grid, posterior, color=color, linewidth=2.5)
    ax.set_title(name, color=color, fontsize=12, pad=10)
    combined.plot(p_grid, posterior, color=color, linewidth=2.5, label=name)
    comparison.append({
        'Prior': name,
        'Posterior mean p': (aa + wins) / (aa + bb + plays),
        'Probability p is profitable': beta.sf(break_even, aa + wins, bb + losses),
    })

fig.suptitle(f'Same {plays} plays, three starting beliefs\n'
             'Top panels: dashed = prior, solid = posterior', fontsize=15)
top_axes[0].set_ylabel('Density')
for ax in top_axes[1:]:
    ax.tick_params(labelleft=False)
combined.set_title('Compare the three posteriors', loc='left', fontsize=14, pad=12)
combined.set_ylabel('Posterior density')

for ax in [*top_axes, combined]:
    ax.axvline(break_even, color='#C05252', linestyle=':', linewidth=1.6,
               label=f'Break even ({break_even:.1%})')
    ax.set(xlim=(0, 1), ylim=(0, None), xlabel='Win probability p')
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.grid(axis='y', alpha=.15)
    ax.set_axisbelow(True)

combined.legend(loc='upper center', bbox_to_anchor=(.5, -.17),
                ncol=4, frameon=False, fontsize=10)
plt.show()
display(pd.DataFrame(comparison).round(3))


## 5. Watch the posterior evolve
We can reconstruct our belief after each revealed play. The graph below begins with the prior at play zero, then updates it after every win or loss. It uses only results up to `PLAYS_SEEN`.

**Pause and predict:** will every additional play move the estimate in the same direction? Will all three priors still matter equally after hundreds of plays?

A win increases $a$ by one; a loss increases $b$ by one.

The posterior after one batch can become the prior for the next batch. In that case, one must update only with the **new** results. Counting the earlier results again would use the same evidence twice. Here we always start from the original prior and count each revealed play once.


In [ ]:
checkpoints = np.arange(plays+1)
cumulative_wins = np.r_[0, observed.win.cumsum().to_numpy()]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, (aa, bb) in PRIOR_CHOICES.items():
    updated_a = aa+cumulative_wins
    updated_b = bb+checkpoints-cumulative_wins
    axes[0].plot(checkpoints, updated_a/(updated_a+updated_b), label=name)
low, high = beta.ppf(.05, a+cumulative_wins, b+checkpoints-cumulative_wins), beta.ppf(.95, a+cumulative_wins, b+checkpoints-cumulative_wins)
axes[1].fill_between(checkpoints, low, high, alpha=.25, label='90% posterior interval')
axes[1].plot(checkpoints, (a+cumulative_wins)/(a+b+checkpoints), label='Posterior mean')
axes[0].set_title('Same observations, three starting beliefs')
axes[1].set_title(f'Uncertainty with {SELECTED_PRIOR}')
for ax in axes:
    ax.axhline(break_even, color='tomato', linestyle='--', label='Break even')
    ax.set_xlabel('Number of revealed plays'); ax.set_ylabel('Win probability p'); ax.set_ylim(0, 1); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


The posterior mean is a weighted average of the prior mean and the observed win fraction. The prior gets weight $a+b$; the observations get weight equal to the number of plays. More data gradually reduce the relative influence of the starting belief.

The interval usually becomes narrower as evidence accumulates, but an individual surprising result need not narrow it. More observations also do not guarantee that the estimate moves steadily in one direction.

To see snapshots of the whole curve, compare the prior with the posteriors at a few points in the run below.


In [ ]:
snapshots = sorted(set([0, min(5, plays), min(20, plays), min(100, plays), plays]))
for n in snapshots:
    w = int(observed.win.iloc[:n].sum())
    plt.plot(p_grid, beta.pdf(p_grid, a+w, b+n-w), label=f'After {n} plays')
plt.axvline(break_even, color='tomato', linestyle='--')
plt.xlabel('Possible p'); plt.ylabel('Density'); plt.legend(); plt.show()


## 6. Would you keep playing?
A profitable win probability and a profitable short run are different things.

For a possible $p$, the expected profit per play is $2.4p - 1$. Even if that is positive, the next 100 plays can still lose money. We have two uncertainties: we do not know $p$, and the plays themselves are random even if $p$ were known.

To describe the future, we can:

1. Draw a possible $p$ from our posterior.
2. Simulate 100 new plays using that $p$.
3. Record the profit, then repeat.

This is a **posterior predictive simulation**. The Beta posterior is easy to sample directly; the optional section below shows how HMC can obtain samples too.

**Pause and predict:** if you become confident about $p$, does that make a short run of plays risk-free?


In [ ]:
rng = np.random.default_rng(21)
p_samples = rng.beta(post_a, post_b, size=10000)
future_profit = simulate_profit(p_samples)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(p_samples, bins=30, density=True); axes[0].axvline(break_even, color='tomato', linestyle='--')
axes[0].set_xlabel('Possible p from the posterior'); axes[0].set_ylabel('Density')
axes[1].hist(future_profit, bins=30); axes[1].axvline(0, color='tomato', linestyle='--')
axes[1].set_xlabel('Simulated profit over the NEXT 100 plays'); axes[1].set_ylabel('Number of simulations')
plt.tight_layout(); plt.show()
print(f'Probability p is profitable: {beta.sf(break_even, post_a, post_b):.1%}')
print(f'Mean expected profit per play: {2.4*post_a/(post_a+post_b)-1:.3f} tokens')
print(f'Probability of profit in the next 100 plays: {np.mean(future_profit > 0):.1%}')
print('Middle 90% of simulated profits:', np.quantile(future_profit, [.05, .95]).round(1))


Write down whether you would play another 100 times, and why. Someone trying to maximize expected profit may choose differently from someone who wants to limit the chance of a loss.

Now return to `PLAYS_SEEN` and reveal the next batch. Rerun the cells below it. Does your decision change after 20, 100, or 500 observations? Would it change under a different prior?

Finish these sentences with your team:

- Before playing, our prior expressed …
- After observing the plays, our posterior suggested …
- We would / would not keep playing because …
- We would reconsider if …

**Connection to AI:** observing a few successful predictions is not the same as knowing a model's true success rate. More evidence can improve our estimate, while future successes and failures remain uncertain.


## Obtain the posterior with Hamiltonian Monte Carlo (HMC) sampling

> Before running this section, make sure you have completed the main path above updating `PLAYS_SEEN` to 500.

Our Beta model has an exact posterior, which made the earlier updates analytical and easy to calculate. More complicated models often are not analytical. **Markov chain Monte Carlo (MCMC)** methods, a class of probability distribution samplers, let us represent a posterior with samples when a direct formula is unavailable.

We will use the **No-U-Turn Sampler (NUTS)**, an adaptive version of HMC, on the same single-machine problem. This is deliberately a problem we can already solve: we can compare the sampler with the exact answer before trusting it on a harder model.

The model is just:

```python
p = pm.Beta('p', alpha=a, beta=b)
pm.Binomial('wins', n=plays, p=p, observed=wins)
```

There is still only one unknown win probability $p$. The first line is our selected prior; the second connects $p$ to the observed win count.


In [ ]:
import arviz as az
USE_CACHED = False
a, b = PRIOR_CHOICES[SELECTED_PRIOR]
model, trace = fit_slot(wins, plays, a=a, b=b)
nuts_samples = trace.posterior.p.values.ravel()
# Final reveal: this is the fixed probability used to generate the plays.
# It was not used to choose the prior or fit the posterior.
TRUE_WIN_RATE = 0.50

fig, ax = plt.subplots(figsize=(12, 5), layout='constrained')
ax.hist(nuts_samples, bins=25, density=True, alpha=.4,
        color='#2878B5', label='NUTS samples')
ax.plot(p_grid, beta.pdf(p_grid, a+wins, b+losses),
        color='#2878B5', linewidth=2.5, label='Exact posterior')
ax.axvline(TRUE_WIN_RATE, color='#222222', linewidth=2.5,
           label=f'True win rate ({TRUE_WIN_RATE:.0%})')
ax.axvline(break_even, color='#C05252', linestyle=':', linewidth=1.8,
           label=f'Break even ({break_even:.1%})')
ax.set(xlim=(0, 1), ylim=(0, None), xlabel='Win probability p', ylabel='Density',
       title=f'Final reveal: posterior after {plays} plays')
from matplotlib.ticker import PercentFormatter
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.grid(axis='y', alpha=.15)
ax.set_axisbelow(True)
ax.legend(loc='upper center', bbox_to_anchor=(.5, -.15), ncol=2, frameon=False)
plt.show()
print('The true win rate is 50%: every play is a fair coin flip.')
print()
print('Our fictional casino keeps a handful of these machines around to boost gambler morale.')
print('Most of its machines take your tokens in the long run. These ones come out of the marketing budget.')
print()
print('Fair coin flips, generous payouts: at 2.4 tokens per win, this machine gives the player')
print('an expected profit of 0.20 tokens per play. A financially fair game would break even at 41.7%.')
print('Mean from NUTS:', round(nuts_samples.mean(), 3))
print('Exact mean:', round((a+wins)/(a+b+plays), 3))


### Check the sampler before using its output
NUTS runs several sequences of samples called **chains**. We want the chains to explore the same posterior, rather than getting stuck in different regions.

- **R-hat** compares chains. Values close to 1 are reassuring; above about 1.01 deserves investigation.
- **ESS (effective sample size)** accounts for the fact that neighboring samples can be similar. More effective samples allow more precise estimates.
- **Divergences** warn that the sampler had difficulty exploring. We aim for zero.

The trace plot shows the sampled $p$ values within each chain. The histogram above should also agree with the exact posterior, apart from finite-sample variation.

These checks concern the calculation. They cannot establish that the prior is reasonable or that the assumption of a fixed win probability is appropriate.


In [ ]:
display(az.summary(trace, var_names=['p'], round_to=3))
print('Divergences:', int(trace.sample_stats.diverging.sum()))
az.plot_trace(trace, var_names=['p']); plt.tight_layout(); plt.show()
